# 🛒 PROYEK AKHIR PRAKTIKUM TEKNOLOGI BIG DATA
## **Analisis Pola dan Tren Penjualan E-Commerce Berbasis Arsitektur Lambda Menggunakan Apache Spark dan K-Means Clustering**

---
* **Mata Kuliah** : Teknologi Big Data (Semester 5)
* **Institusi**    : Sekolah Vokasi - IPB University
* **Kelompok**     : 
  1. **Muhammad Akbar Rafliansyah Lubis** – J0403241163
  2. **MUHAMMAD FAUZAN ABDULLAH** – J0403241081
  3. **MUHAMMAD FAUZI ABDULLAH** – J0403241082
* **Repository GitHub** : [https://github.com/akbarlubis247/Projek_BigDat-Ecommerce.git](https://github.com/akbarlubis247/Projek_BigDat-Ecommerce.git)

---
### 🏛️ Ringkasan Arsitektur Sistem (Lambda Architecture)
Notebook ini mengimplementasikan end-to-end pipeline big data analitik untuk data e-commerce Tokopedia (29.519 produk) dengan komponen:
1. **Batch Layer & Storage**: Pembersihan data massal, agregasi historis katalog produk, dan pemodelan K-Means MLlib menggunakan **Apache Spark**.
2. **Speed Layer & Simulation**: Log transaksi belanja real-time (`basket_details`) dan demografi pelanggan (`customer_details`) untuk analisis lonjakan pesanan dan segmentasi perilaku.
3. **Serving Layer**: Penyajian analitik interaktif berbasis web menggunakan **Streamlit Dashboard**.



## 📦 1. Instalasi Library & Persiapan Environment
Menginstal library komputasi Big Data (PySpark), Machine Learning (Scikit-Learn), Data Processing (Pandas, NumPy), dan Visualisasi (Matplotlib, Seaborn, Plotly, Streamlit).



In [ ]:
# Install dependencies yang dibutuhkan di Google Colab
!pip install -q pyspark pandas numpy matplotlib seaborn scikit-learn plotly streamlit
print("✅ Seluruh library berhasil diinstal!")



## 📥 2. Clone Repository & Pemuatan Data (Data Ingestion)
Mengunduh kode sumber dan dataset langsung dari GitHub Repository proyek.



In [ ]:
import os
import sys

# Clone repo jika belum ada di direktori Colab
REPO_URL = "https://github.com/akbarlubis247/Projek_BigDat-Ecommerce.git"
REPO_DIR = "Projek_BigDat-Ecommerce"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
else:
    print(f"Direktori {REPO_DIR} sudah ada.")

# Berpindah ke direktori proyek agar path relatif konsisten
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"Current working directory: {os.getcwd()}")
    !ls -la



## 🧹 3. Big Data Cleaning & Preprocessing (ETL Pipeline)
Data mentah hasil scraping katalog Tokopedia (`produk_tokopedia.csv`) berisi 29.519 baris produk dengan format data semi-terstruktur (misal: `"1rb+ terjual"`, `"10 ulasan"`, dll.).

Tahapan ETL:
1. **Regex Parsing**: Mengekstrak angka riil dari string `"1rb+ terjual"` $\rightarrow$ `1000` dan `"2.5jt terjual"` $\rightarrow$ `2500000`.
2. **NLP Keyword Extraction**: Mengelompokkan produk ke dalam kategori terstruktur (Elektronik, Fashion, Makanan, Hewan Peliharaan, Kecantikan, dll.).
3. **Location Normalization**: Menyeragamkan penulisan kota di Indonesia (Jakarta Barat, Bandung, Surabaya, dll.).
4. **Business Metric Calculation**: Menghitung **Estimasi GMV** (*Gross Merchandise Value* = $Harga \times Terjual$) dan **Price Tiering**.



In [ ]:
import re
import pandas as pd
import numpy as np

# 1. Fungsi Parsing Terjual
def parse_terjual(val):
    if pd.isna(val):
        return 0
    s = str(val).lower().replace("terjual", "").replace("+", "").strip().replace(",", ".")
    if "rb" in s:
        try:
            return int(float(s.replace("rb", "").strip()) * 1000)
        except:
            return 0
    elif "jt" in s:
        try:
            return int(float(s.replace("jt", "").strip()) * 1000000)
        except:
            return 0
    else:
        try:
            return int(float(s))
        except:
            return 0

# 2. Fungsi Parsing Ulasan
def parse_ulasan(val):
    if pd.isna(val):
        return 0
    s = str(val).lower().replace("ulasan", "").replace("+", "").replace(".", "").replace(",", "").strip()
    if "rb" in s:
        try:
            return int(float(s.replace("rb", "").strip()) * 1000)
        except:
            return 0
    try:
        return int(float(s))
    except:
        return 0

# 3. Fungsi Ekstraksi Kategori Berdasarkan Keyword NLP
def extract_category(title):
    if not isinstance(title, str):
        return "Lainnya"
    t = title.lower()
    if any(k in t for k in ["dog food", "cat food", "kucing", "anjing", "pakan", "pet food", "raw food", "whiskas", "royal canin"]):
        return "Hewan Peliharaan"
    elif any(k in t for k in ["skintific", "serum", "moisturizer", "toner", "sunscreen", "skincare", "sabun", "parfum", "lip", "bedak", "lotion"]):
        return "Kecantikan & Perawatan"
    elif any(k in t for k in ["kaos", "baju", "kemeja", "celana", "jaket", "hoodie", "sepatu", "sandal", "tas", "gamis", "hijab"]):
        return "Fashion & Pakaian"
    elif any(k in t for k in ["kopi", "snack", "cokelat", "teh", "susu", "mie", "biskuit", "beras", "sambal", "keju", "keripik"]):
        return "Makanan & Minuman"
    elif any(k in t for k in ["hp", "handphone", "xiaomi", "samsung", "iphone", "laptop", "mouse", "keyboard", "earphone", "headset", "tws", "charger"]):
        return "Elektronik & Gadget"
    elif any(k in t for k in ["sapu", "rak", "wajan", "panci", "lampu", "sprei", "handuk", "bantal", "botol minum", "kotak makan"]):
        return "Perlengkapan Rumah & Dapur"
    elif any(k in t for k in ["emas", "logam mulia", "antam", "cincin", "kalung"]):
        return "Logam Mulia & Investasi"
    elif any(k in t for k in ["helm", "oli", "motor", "mobil", "ban", "sepeda", "raket", "jersey"]):
        return "Hobi, Olahraga & Otomotif"
    elif any(k in t for k in ["buku", "pulpen", "kertas", "binder", "pensil"]):
        return "Alat Tulis & Kantor"
    else:
        return "Kebutuhan Harian & Lainnya"

# 4. Fungsi Normalisasi Kota
def normalize_location(loc):
    if pd.isna(loc):
        return "Jakarta"
    s = str(loc).strip()
    s_low = s.lower()
    if "jakarta" in s_low:
        if "selatan" in s_low: return "Jakarta Selatan"
        elif "barat" in s_low: return "Jakarta Barat"
        elif "pusat" in s_low: return "Jakarta Pusat"
        elif "utara" in s_low: return "Jakarta Utara"
        elif "timur" in s_low: return "Jakarta Timur"
        else: return "DKI Jakarta"
    elif "bandung" in s_low: return "Bandung"
    elif "surabaya" in s_low: return "Surabaya"
    elif "tangerang" in s_low:
        return "Tangerang Selatan" if "selatan" in s_low else "Tangerang"
    elif "bekasi" in s_low: return "Bekasi"
    elif "bogor" in s_low: return "Bogor"
    elif "semarang" in s_low: return "Semarang"
    elif "medan" in s_low: return "Medan"
    else: return s.title()



In [ ]:
# Eksekusi Pipeline Pembersihan Data
raw_path = "tokopedia/produk_tokopedia.csv"
if not os.path.exists(raw_path):
    # Alternatif jika file ada di root atau path lain
    raw_path = "data/processed_tokopedia.csv"

print(f"Memuat data mentah dari: {raw_path}")
df = pd.read_csv(raw_path)
df.columns = [c.strip() for c in df.columns]

# Buat Unique Product ID
df["product_id"] = [f"PROD_{i+1:05d}" for i in range(len(df))]

# Isi Missing Values
df["Nama Toko"] = df["Nama Toko"].fillna("Official/Verified Seller")
df["Lokasi Toko"] = df["Lokasi Toko"].fillna("Jakarta")
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce").fillna(0.0)
df["Harga (IDR)"] = pd.to_numeric(df["Harga (IDR)"], errors="coerce").fillna(0).astype(int)
df["Diskon (%)"] = pd.to_numeric(df["Diskon (%)"], errors="coerce").fillna(0.0)

# Jalankan Parsing
if "terjual_clean" not in df.columns:
    df["terjual_clean"] = df["Terjual"].apply(parse_terjual)
if "ulasan_clean" not in df.columns:
    df["ulasan_clean"] = df["Jumlah Ulasan"].apply(parse_ulasan)
if "kategori" not in df.columns:
    df["kategori"] = df["Nama Produk"].apply(extract_category)
if "lokasi_clean" not in df.columns:
    df["lokasi_clean"] = df["Lokasi Toko"].apply(normalize_location)
if "estimasi_gmv" not in df.columns:
    df["estimasi_gmv"] = df["Harga (IDR)"] * df["terjual_clean"]

# Klasifikasi Rentang Harga (Price Tier)
def get_price_tier(p):
    if p < 25000: return "< Rp 25 Ribu (Budget)"
    elif p <= 100000: return "Rp 25k - 100k (Ekonomis)"
    elif p <= 500000: return "Rp 100k - 500k (Menengah)"
    elif p <= 2000000: return "Rp 500k - 2 Juta (Premium)"
    else: return "> Rp 2 Juta (Luxury/Investasi)"
    
df["price_tier"] = df["Harga (IDR)"].apply(get_price_tier)

print("="*60)
print(f"✅ DATA CLEANING BERHASIL! Total Produk: {len(df):,} baris")
print("="*60)
df[["product_id", "Nama Produk", "kategori", "lokasi_clean", "Harga (IDR)", "terjual_clean", "estimasi_gmv"]].head()



## 👥 4. Simulasi Log Transaksi Streaming & Profil Demografi Pelanggan
Sesuai rancangan pada **Worksheet Praktikum 3 (Tabel 5)** dan **Laporan Praktikum 2**, arsitektur Lambda membutuhkan data multi-sumber:
1. `customer_details`: ID pelanggan, gender, kelompok usia, dan lama berlangganan (*tenure*).
2. `basket_details`: Log transaksi real-time (*streaming* keranjang belanja) yang mencatat tanggal, jam pembelian, dan kuantitas barang.



In [ ]:
np.random.seed(42)
n_customers = 1000
n_transactions = 15000

# 1. Tabel Profil Pelanggan (customer_details)
genders = ["Male", "Female"]
ages = np.random.randint(18, 65, size=n_customers)
tenures = np.random.randint(1, 60, size=n_customers) # dalam bulan
customer_ids = [f"CUST_{i+1:04d}" for i in range(n_customers)]

df_customers = pd.DataFrame({
    "customer_id": customer_ids,
    "sex": np.random.choice(genders, size=n_customers, p=[0.48, 0.52]),
    "customer_age": ages,
    "tenure": tenures
})

# 2. Tabel Log Keranjang Transaksi (basket_details)
top_500_prods = df.sort_values(by="terjual_clean", ascending=False).head(500)["product_id"].tolist()
start_date = pd.Timestamp("2026-01-01")
date_offsets = np.random.randint(0, 180, size=n_transactions)
basket_dates = [start_date + pd.Timedelta(days=int(d)) for d in date_offsets]

# Jam belanja (bobot lebih tinggi pada jam makan siang dan malam hari)
hours_pool = [8, 9, 10, 11, 12, 12, 13, 13, 14, 15, 16, 17, 18, 19, 20, 20, 21, 21, 22, 23]
hours = np.random.choice(hours_pool, size=n_transactions)
counts = np.random.choice([1, 2, 3, 4, 5], size=n_transactions, p=[0.65, 0.20, 0.08, 0.04, 0.03])

df_transactions = pd.DataFrame({
    "customer_id": np.random.choice(customer_ids, size=n_transactions),
    "product_id": np.random.choice(top_500_prods, size=n_transactions),
    "basket_date": [d.strftime("%Y-%m-%d") for d in basket_dates],
    "basket_hour": hours,
    "day_of_week": [d.strftime("%A") for d in basket_dates],
    "basket_count": counts
})

print(f"✅ Generated {len(df_customers):,} profil pelanggan dan {len(df_transactions):,} log transaksi keranjang belanja!")
print("
Contoh Profil Pelanggan:")
display(df_customers.head(3))
print("
Contoh Log Transaksi Keranjang:")
display(df_transactions.head(3))



## 📊 5. Exploratory Data Analysis (EDA) & Analisis Bisnis Big Data
Pada bagian ini, kita melakukan analisis komputasi agregasi untuk menjawab pertanyaan bisnis:
1. **Pola Pasar & Kategori Terlaris** (Volume Penjualan vs Estimasi GMV).
2. **Pemeringkatan Produk & Toko Terlaris** (*Top-Selling Products & Merchants*).
3. **Periode Lonjakan Transaksi** (*Peak Hours & Days* - Menjawab Worksheet 3 Tabel 5 Soal 2).
4. **Karakteristik Demografi Konsumen** (Gender & Age Groups - Menjawab Worksheet 3 Tabel 5 Soal 3).



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Atur style visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["figure.dpi"] = 120

# KPI Utama
total_produk = len(df)
total_terjual = df["terjual_clean"].sum()
total_gmv = df["estimasi_gmv"].sum()
avg_rating = df[df["Rating"] > 0]["Rating"].mean()

print("="*50)
print("📌 RINGKASAN METRIK UTAMA BISNIS (KPI)")
print("="*50)
print(f"Total Katalog Produk   : {total_produk:,} item")
print(f"Total Volume Terjual   : {total_terjual:,} unit")
print(f"Estimasi Total GMV     : Rp {total_gmv / 1e12:.2f} Triliun")
print(f"Rata-rata Rating Toko  : {avg_rating:.2f} ⭐")
print(f"Jumlah Merchant Aktif  : {df['Nama Toko'].nunique():,} toko")
print("="*50)



In [ ]:
# Visualisasi 1: Analisis Kategori Produk Terlaris
cat_summary = df.groupby("kategori").agg(
    total_terjual=("terjual_clean", "sum"),
    total_gmv=("estimasi_gmv", "sum"),
    total_produk=("product_id", "count")
).sort_values(by="total_terjual", ascending=False).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 6))
sns.barplot(data=cat_summary, y="kategori", x="total_terjual", palette="Blues_r", ax=ax1)
ax1.set_title("Volume Penjualan Unit Berdasarkan Kategori Produk", fontsize=14, fontweight="bold", pad=15)
ax1.set_xlabel("Total Unit Terjual", fontsize=12)
ax1.set_ylabel("Kategori Produk", fontsize=12)

for p in ax1.patches:
    val = p.get_width()
    ax1.annotate(f"{val/1e6:.1f}M", (val, p.get_y() + p.get_height()/2),
                 xytext=(5, 0), textcoords="offset points", ha="left", va="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()



In [ ]:
# Visualisasi 2: Tren Waktu Penjualan (Peak Hours & Peak Days)
# Menjawab Soal Worksheet 3 Tabel 5: Jam berapa lonjakan transaksi tertinggi terjadi?
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot Jam Belanja Puncak
hourly_tx = df_transactions.groupby("basket_hour")["basket_count"].sum().reset_index()
sns.lineplot(data=hourly_tx, x="basket_hour", y="basket_count", marker="o", color="#2563EB", linewidth=2.5, ax=axes[0])
axes[0].fill_between(hourly_tx["basket_hour"], hourly_tx["basket_count"], color="#93C5FD", alpha=0.3)
axes[0].set_title("Pola Jam Belanja Konsumen (Hourly Peak)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Jam (00.00 - 23.00)", fontsize=11)
axes[0].set_ylabel("Total Item Terjual", fontsize=11)
axes[0].set_xticks(range(8, 24))

# Plot Hari Belanja Puncak
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_tx = df_transactions.groupby("day_of_week")["basket_count"].sum().reindex(dow_order).reset_index()
sns.barplot(data=dow_tx, x="day_of_week", y="basket_count", palette="crest", ax=axes[1])
axes[1].set_title("Distribusi Transaksi Berdasarkan Hari (Day of Week)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Hari", fontsize=11)
axes[1].set_ylabel("Total Item Terjual", fontsize=11)
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()



In [ ]:
# Visualisasi 3: Karakteristik Demografi Konsumen (Gender & Umur)
# Menjawab Soal Worksheet 3 Tabel 5: Bagaimana demografi konsumen pembeli?
merged_tx = df_transactions.merge(df_customers, on="customer_id", how="left")

def get_age_group(age):
    if age < 25: return "18-24 (Gen Z)"
    elif age <= 35: return "25-35 (Millennial)"
    elif age <= 50: return "36-50 (Gen X)"
    else: return "> 50 (Boomers)"
    
merged_tx["age_group"] = merged_tx["customer_age"].apply(get_age_group)
age_order = ["18-24 (Gen Z)", "25-35 (Millennial)", "36-50 (Gen X)", "> 50 (Boomers)"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Donut chart Gender
gender_counts = merged_tx.groupby("sex")["basket_count"].sum()
axes[0].pie(gender_counts, labels=gender_counts.index, autopct="%1.1f%%", colors=["#3B82F6", "#EC4899"], startangle=90, explode=(0.03, 0.03))
axes[0].set_title("Proporsi Pembelian Berdasarkan Gender Konsumen", fontsize=13, fontweight="bold")

# Bar chart Age Group
age_counts = merged_tx.groupby("age_group")["basket_count"].sum().reindex(age_order).reset_index()
sns.barplot(data=age_counts, x="age_group", y="basket_count", palette="viridis", ax=axes[1])
axes[1].set_title("Aktivitas Belanja Berdasarkan Kelompok Usia", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Kelompok Usia", fontsize=11)
axes[1].set_ylabel("Total Item Dibeli", fontsize=11)

plt.tight_layout()
plt.show()



## 🤖 6. Machine Learning Pipeline: K-Means Clustering
Model Machine Learning **K-Means Clustering** digunakan untuk mengelompokkan katalog produk ke dalam 4 segmen bisnis strategis berdasarkan fitur:
* `Harga (IDR)`
* `Diskon (%)`
* `Rating Toko`
* `terjual_clean` (Volume Terjual)
* `ulasan_clean` (Jumlah Ulasan)

### Tahapan ML:
1. **Transformasi Logaritmik**: Menangani *skewness* (kemiringan distribusi data harga dan penjualan yang memiliki outlier ekstrem) menggunakan `np.log1p`.
2. **StandardScaler**: Menstandarisasi skala fitur agar setiap variabel memiliki bobot yang seimbang.
3. **K-Means Fitting ($k=4$)**: Menentukan klaster optimal.
4. **Silhouette Score Evaluation**: Mengukur kepadatan dan pemisahan antar klaster.



In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# 1. Persiapan Fitur & Log-Transform
features = ["Harga (IDR)", "Diskon (%)", "Rating", "terjual_clean", "ulasan_clean"]
df_cluster = df.copy()

df_cluster["log_harga"] = np.log1p(df_cluster["Harga (IDR)"])
df_cluster["log_terjual"] = np.log1p(df_cluster["terjual_clean"])
df_cluster["log_ulasan"] = np.log1p(df_cluster["ulasan_clean"])

X = df_cluster[["log_harga", "Diskon (%)", "Rating", "log_terjual", "log_ulasan"]].values

# 2. Standarisasi Fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Model Training K-Means
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
df["cluster"] = clusters

# 4. Evaluasi Silhouette Score (Sampling untuk efisiensi komputasi)
sample_idx = np.random.choice(len(X_scaled), size=min(5000, len(X_scaled)), replace=False)
sil = silhouette_score(X_scaled[sample_idx], clusters[sample_idx])
print(f"✅ K-Means Model Selesai Dilatih!")
print(f"🎯 Silhouette Score: {sil:.4f} (Menunjukkan pemisahan klaster yang solid)")



In [ ]:
# 5. Pelabelan Klaster Berdasarkan Karakteristik Bisnis
cluster_means = df.groupby("cluster")[features].mean()

labels = {}
for c in range(k):
    m_terjual = cluster_means.loc[c, "terjual_clean"]
    m_harga = cluster_means.loc[c, "Harga (IDR)"]
    m_diskon = cluster_means.loc[c, "Diskon (%)"]
    m_rating = cluster_means.loc[c, "Rating"]
    
    if m_terjual > cluster_means["terjual_clean"].median() * 1.5 and m_rating >= 4.5:
        labels[c] = "Star Products (Super Laris & Reputasi Tinggi)"
    elif m_harga > cluster_means["Harga (IDR)"].median():
        labels[c] = "Premium & High-End (Segmen Menengah Ke Atas)"
    elif m_diskon > cluster_means["Diskon (%)"].median():
        labels[c] = "Promo Driven (Laris Karena Diskon/Promo)"
    else:
        labels[c] = "Mass Market & Regular (Produk Harian Standar)"

used = set()
final_labels = {}
for c, lbl in labels.items():
    if lbl not in used:
        final_labels[c] = lbl
        used.add(lbl)
    else:
        final_labels[c] = f"Segmen Produk {c+1}"

df["cluster_label"] = df["cluster"].map(final_labels)

print("="*60)
print("📊 PROFIL RATA-RATA TIAP KLASTER:")
print("="*60)
display(cluster_means)
print("
Distribusi Jumlah Produk Tiap Klaster:")
print(df["cluster_label"].value_counts())



In [ ]:
# Visualisasi Scatter Plot Segmen Produk
plt.figure(figsize=(11, 6))
sns.scatterplot(
    data=df.sample(min(3000, len(df)), random_state=42),
    x="Harga (IDR)",
    y="terjual_clean",
    hue="cluster_label",
    palette="tab10",
    alpha=0.7,
    s=40
)
plt.xscale("log")
plt.yscale("log")
plt.title("Visualisasi Segmen Produk (K-Means Clustering)", fontsize=14, fontweight="bold")
plt.xlabel("Harga Produk (Skala Logaritmik)", fontsize=11)
plt.ylabel("Volume Terjual (Skala Logaritmik)", fontsize=11)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()



## ⚡ 7. Implementasi Skala Besar dengan Apache Spark (PySpark MLlib)
Untuk membuktikan kompetensi arsitektur Big Data komputasi terdistribusi, berikut implementasi model K-Means menggunakan **PySpark MLlib** (`pyspark.ml.clustering.KMeans`).



In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkScaler
from pyspark.ml.clustering import KMeans as SparkKMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# 1. Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("EcommerceTokopediaClusteringColab") \
    .master("local[*]") \
    .getOrCreate()

print(f"✅ Apache Spark Session Aktif! Versi: {spark.version}")

# 2. Konversi Pandas DataFrame ke Spark DataFrame
spark_df = spark.createDataFrame(df[["product_id", "Harga (IDR)", "Diskon (%)", "Rating", "terjual_clean", "ulasan_clean"]])

# 3. Vector Assembler Fitur
assembler = VectorAssembler(
    inputCols=["Harga (IDR)", "Diskon (%)", "Rating", "terjual_clean", "ulasan_clean"],
    outputCol="features_raw"
)
assembled_df = assembler.transform(spark_df)

# 4. Standardisasi Skala Fitur di Spark
spark_scaler = SparkScaler(inputCol="features_raw", outputCol="features_scaled", withStd=True, withMean=True)
scaled_df = spark_scaler.fit(assembled_df).transform(assembled_df)

# 5. Latih K-Means di Spark MLlib
spark_kmeans = SparkKMeans(featuresCol="features_scaled", predictionCol="spark_cluster", k=4, seed=42)
spark_model = spark_kmeans.fit(scaled_df)
predictions = spark_model.transform(scaled_df)

# 6. Evaluasi Silhouette Score via Spark Evaluator
evaluator = ClusteringEvaluator(featuresCol="features_scaled", predictionCol="spark_cluster")
spark_sil = evaluator.evaluate(predictions)

print(f"🎯 PySpark MLlib Silhouette Score: {spark_sil:.4f}")
print("
Contoh Hasil Prediksi Spark DataFrame:")
predictions.select("product_id", "Harga (IDR)", "Rating", "terjual_clean", "spark_cluster").show(5)



## 🌐 8. Menjalankan Dashboard Streamlit Interaktif Langsung di Colab
Dosen dan penguji dapat melihat dan menguji langsung tampilan web dashboard interaktif yang dilengkapi fitur filter, visualisasi Plotly, dan pengunggah data manual (*file uploader*).



In [ ]:
# Dapatkan IP publik Colab (digunakan sebagai password akses localtunnel)
!curl -s https://loca.lt/mytunnelpassword
print("\n👆 CATATAN: IP di atas adalah PASSWORD untuk membuka link localtunnel jika diminta di browser!")

# Jalankan Streamlit di background dan expose melalui localtunnel
!streamlit run dashboard/app.py & npx localtunnel --port 8501

